<a href="https://colab.research.google.com/github/hhw215/Computer-Vision-Project/blob/main/cv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [1]:
import sys
!pip install -U timm pandas pillow numpy torchvision

In [2]:
import copy
import csv
import json
import random
import time
from collections.abc import Iterable
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Dict, List, Optional, Sequence, Tuple
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Globals

In [4]:
BASE_DIR = "/content/drive/MyDrive/CV"
CONFIG = {
    "seed": 42,
    "image_size": 224,
    "model_name": "deit_small_patch16_224", # for time reason used only one backbone
    "pretrained": True, # pretrained=True means that the model is initialized with weights that were already trained on a dataset, rather than starting from random weights.
    "attention_type": "moh", # "standard", "moh", "pyra", "meta"
    "top_k": 4, #"none", 4
    "replace_layers": "all", # "none", "all", "6, 7, 8, 9, 10, 11", "0, 1, 2, 3, 4, 5", "0, 2, 4, 6, 8, 10"
    "batch_size": 16,
    "num_workers": 2,
    "epochs": 3,
    "lr": 1e-5,
    "router_lr": 3e-5,
    "weight_decay": 0.05,
    "warmup_ratio": 0.05,
    "min_lr": 1e-7,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "train_csv": f"{BASE_DIR}/dataset/splits/train.csv",
    "val_csv": f"{BASE_DIR}/dataset/splits/val.csv",
    "test_csv": f"{BASE_DIR}/dataset/splits/test.csv",
    "images_root": f"{BASE_DIR}/dataset/raw",
    "triplets_csv": f"{BASE_DIR}/dataset/processed/triplet_available.csv",
    "run_name": "template_run",
    "out_dir": f"{BASE_DIR}/outputs",
}
print(json.dumps(CONFIG, indent=2))
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

{
  "seed": 42,
  "image_size": 224,
  "model_name": "deit_small_patch16_224",
  "pretrained": true,
  "attention_type": "moh",
  "top_k": 4,
  "replace_layers": "all",
  "batch_size": 16,
  "num_workers": 2,
  "epochs": 3,
  "lr": 1e-05,
  "router_lr": 3e-05,
  "weight_decay": 0.05,
  "warmup_ratio": 0.05,
  "min_lr": 1e-07,
  "device": "cuda",
  "train_csv": "/content/drive/MyDrive/CV/dataset/splits/train.csv",
  "val_csv": "/content/drive/MyDrive/CV/dataset/splits/val.csv",
  "test_csv": "/content/drive/MyDrive/CV/dataset/splits/test.csv",
  "images_root": "/content/drive/MyDrive/CV/dataset/raw",
  "triplets_csv": "/content/drive/MyDrive/CV/dataset/processed/triplet_available.csv",
  "run_name": "template_run",
  "out_dir": "/content/drive/MyDrive/CV/outputs"
}
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


# Utils

In [5]:
REQUIRED_TRIPLET_COLUMNS = (
    "reference",
    "candidate_a",
    "candidate_b",
    "choice",
)
class NightsTripletDataset(Dataset):
    def __init__(self, split_csv: str, images_root: Optional[str] = None, transform=None) -> None:
        self.split_csv = Path(split_csv)
        self.images_root = Path(images_root) if images_root else None
        self.transform = transform
        df = pd.read_csv(self.split_csv)
        missing = [column for column in REQUIRED_TRIPLET_COLUMNS if column not in df.columns]
        if missing:
            raise ValueError(f"Split CSV is missing required columns {missing}. Found: {list(df.columns)}")
        self.df = df
    def __len__(self) -> int:
        return len(self.df)
    def __getitem__(self, idx: int) -> Dict:
        row = self.df.iloc[idx]
        ref_path = self._resolve_path(row["reference"])
        a_path = self._resolve_path(row["candidate_a"])
        b_path = self._resolve_path(row["candidate_b"])
        ref_img = self._load_image(ref_path)
        a_img = self._load_image(a_path)
        b_img = self._load_image(b_path)
        if self.transform is not None:
            ref_img = self.transform(ref_img)
            a_img = self.transform(a_img)
            b_img = self.transform(b_img)
        label = int(row["choice"])
        return {
            "reference": ref_img,
            "candidate_a": a_img,
            "candidate_b": b_img,
            "choice": label,
            "reference_path": str(ref_path),
            "candidate_a_path": str(a_path),
            "candidate_b_path": str(b_path),
        }
    def _resolve_path(self, raw_path: str) -> Path:
        path = Path(str(raw_path))
        if path.is_absolute():
            return path
        if self.images_root is None:
            return path
        return self.images_root / path
    @staticmethod
    def _load_image(path: Path) -> Image.Image:
        with Image.open(path) as image:
            return image.convert("RGB")
    def check_missing_files(self) -> pd.DataFrame:
        rows = []
        for index, row in self.df.iterrows():
            ref_path = self._resolve_path(row["reference"])
            a_path = self._resolve_path(row["candidate_a"])
            b_path = self._resolve_path(row["candidate_b"])
            if not (ref_path.exists() and a_path.exists() and b_path.exists()):
                rows.append({
                    "row_index": index,
                    "reference_exists": ref_path.exists(),
                    "candidate_a_exists": a_path.exists(),
                    "candidate_b_exists": b_path.exists(),
                })
        return pd.DataFrame(rows)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
def build_transforms(image_size: int = 224, train: bool = True) -> transforms.Compose:
    resize_size = int((256 / 224) * image_size)
    if train:
        return transforms.Compose([
            transforms.Resize(resize_size, interpolation=InterpolationMode.BICUBIC),
            transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0), interpolation=InterpolationMode.BICUBIC),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize(resize_size, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
@dataclass(frozen=True)
class BackboneConfig:
    model_name: str = "deit_small_patch16_224"
    pretrained: bool = True
    image_size: int = 224
    proj_dim: Optional[int] = None
    drop_path_rate: float = 0.0
    l2_normalize: bool = True
class VisionTransformerEncoder(nn.Module):
    def __init__(self, config: BackboneConfig) -> None:
        super().__init__()
        self.config = config
        self.backbone = timm.create_model(config.model_name, pretrained=config.pretrained, num_classes=0, img_size=config.image_size, drop_path_rate=config.drop_path_rate)
        self.embedding_dim = int(self.backbone.num_features)
        output_dim = config.proj_dim or self.embedding_dim
        self.projection = nn.Identity() if output_dim == self.embedding_dim else nn.Linear(self.embedding_dim, output_dim)
        self.output_dim = output_dim
    def forward_features(self, images: torch.Tensor) -> torch.Tensor:
        features = self.backbone.forward_features(images)
        if isinstance(features, (tuple, list)):
            features = features[0]
        if features.ndim == 3:
            embeddings = features[:, 0]
        elif features.ndim == 2:
            embeddings = features
        else:
            raise ValueError(f"Unsupported feature shape: {tuple(features.shape)}")
        embeddings = self.projection(embeddings)
        if self.config.l2_normalize:
            embeddings = F.normalize(embeddings, dim=-1)
        return embeddings
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.forward_features(images)
AttentionFactory = Callable[[nn.Module, int], nn.Module]
def _resolve_vit_blocks(model: nn.Module):
    backbone = getattr(model, "backbone", model)
    blocks = getattr(backbone, "blocks", None)
    if blocks is None:
        raise AttributeError("Model does not expose a ViT-style 'blocks' attribute.")
    return blocks
def replace_attention_modules(model: nn.Module, attention_factory: AttentionFactory, block_indices: Optional[Iterable[int]] = None) -> List[str]:
    blocks = _resolve_vit_blocks(model)
    if block_indices is None:
        block_indices = range(len(blocks))
    replaced_paths: List[str] = []
    for block_index in block_indices:
        old_attention = blocks[block_index].attn
        blocks[block_index].attn = attention_factory(old_attention, block_index)
        replaced_paths.append(f"blocks.{block_index}.attn")
    return replaced_paths
def parse_replace_layers(value: str) -> str | List[int] | None:
    lowered = value.strip().lower()
    if lowered in {"none", "", "null"}:
        return None
    if lowered == "all":
        return "all"
    return [int(item.strip()) for item in value.split(",") if item.strip()]
def resolve_block_indices(num_blocks: int, replace_layers: str | Iterable[int] | None) -> List[int]:
    if replace_layers is None:
        return []
    if replace_layers == "all":
        return list(range(num_blocks))
    return list(replace_layers)
def make_attention_factory(attention_type: str, top_k: int | None = 4):
    if attention_type == "standard":
        return lambda old_attn, block_idx: old_attn
    if attention_type == "moh":
        return lambda old_attn, block_idx: MoHAttention(old_attn, top_k=top_k)
    if attention_type == "pyra":
        return lambda old_attn, block_idx: PyraAttention(old_attn, top_k=top_k)
    if attention_type == "meta":
        return lambda old_attn, block_idx: MetaAttention(old_attn, top_k=top_k)
    raise ValueError(f"Unknown attention_type: {attention_type}")
@dataclass
class TripletScores:
    sim_a: torch.Tensor
    sim_b: torch.Tensor
    pred: torch.Tensor
class DreamSimLikePipeline(nn.Module):
    def __init__(self, encoder: nn.Module) -> None:
        super().__init__()
        self.encoder = encoder
    def embed(self, images: torch.Tensor) -> torch.Tensor:
        return self.encoder(images)
    def score_triplet(self, reference: torch.Tensor, candidate_a: torch.Tensor, candidate_b: torch.Tensor) -> TripletScores:
        e_ref = self.embed(reference)
        e_a = self.embed(candidate_a)
        e_b = self.embed(candidate_b)
        sim_a = F.cosine_similarity(e_ref, e_a, dim=-1)
        sim_b = F.cosine_similarity(e_ref, e_b, dim=-1)
        pred = torch.where(sim_a >= sim_b, 0, 1).long()
        return TripletScores(sim_a=sim_a, sim_b=sim_b, pred=pred)
    def forward(self, reference: torch.Tensor, candidate_a: torch.Tensor, candidate_b: torch.Tensor) -> Dict[str, torch.Tensor]:
        out = self.score_triplet(reference, candidate_a, candidate_b)
        return {"sim_a": out.sim_a, "sim_b": out.sim_b, "pred": out.pred}
def evaluate(split_csv: str, images_root: str, model_name: str, pretrained: bool, image_size: int, batch_size: int, num_workers: int, device: str, max_batches: int | None = None, encoder: torch.nn.Module | None = None) -> Dict[str, float]:
    transform = build_transforms(image_size=image_size, train=False)
    dataset = NightsTripletDataset(split_csv=split_csv, images_root=images_root, transform=transform)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=device.startswith("cuda"))
    if encoder is None:
        encoder = VisionTransformerEncoder(BackboneConfig(model_name=model_name, pretrained=pretrained, image_size=image_size))
    encoder = encoder.to(device)
    encoder.eval()
    pipeline = DreamSimLikePipeline(encoder).to(device)
    pipeline.eval()
    total = 0
    correct = 0
    elapsed_seconds = 0.0
    batches_ran = 0
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            if max_batches is not None and batch_idx >= max_batches:
                break
            ref = batch["reference"].to(device, non_blocking=True)
            cand_a = batch["candidate_a"].to(device, non_blocking=True)
            cand_b = batch["candidate_b"].to(device, non_blocking=True)
            label = batch["choice"].to(device, non_blocking=True)
            start = time.perf_counter()
            out = pipeline(ref, cand_a, cand_b)
            if device.startswith("cuda"):
                torch.cuda.synchronize()
            elapsed_seconds += time.perf_counter() - start
            pred = out["pred"]
            correct += (pred == label).sum().item()
            total += label.numel()
            batches_ran += 1
    if total == 0:
        raise RuntimeError("No evaluation samples available. Check split CSV and image paths.")
    accuracy = correct / total
    images_processed = total * 3
    ms_per_image = (elapsed_seconds * 1000.0 / images_processed) if images_processed > 0 else float("nan")
    images_per_second = (images_processed / elapsed_seconds) if elapsed_seconds > 0 else float("inf")
    metrics = {
        "samples": total,
        "batches": batches_ran,
        "accuracy_2afc": accuracy,
        "ms_per_image": ms_per_image,
        "images_per_second": images_per_second,
        "elapsed_seconds": elapsed_seconds,
    }
    if device.startswith("cuda"):
        metrics["max_vram_mb"] = torch.cuda.max_memory_allocated() / (1024 ** 2)
    return metrics # Latency and throughput are closely related, but not always exact inverses, especially when using batches.
def _validate_triplet_columns(df: pd.DataFrame) -> None:
    missing = [column for column in REQUIRED_TRIPLET_COLUMNS if column not in df.columns]
    if missing:
        raise ValueError(f"Triplet CSV is missing required columns {missing}. Found: {list(df.columns)}")
def _validate_binary_choice(series: pd.Series) -> pd.Series:
    labels = pd.to_numeric(series, errors="raise").astype(int)
    invalid = sorted(value for value in labels.unique().tolist() if value not in (0, 1))
    if invalid:
        raise ValueError(f"Choice labels must be binary 0/1. Found invalid values: {invalid}")
    return labels
def _stratified_indices(labels: np.ndarray, train_ratio: float, val_ratio: float, test_ratio: float, seed: int, dataset_ratio:float=0.2) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    train_idx: List[int] = []
    val_idx: List[int] = []
    test_idx: List[int] = []
    for label in np.unique(labels):
        cls_indices = np.where(labels == label)[0]
        rng.shuffle(cls_indices)
        n_selected = int(round(len(cls_indices) * dataset_ratio))
        cls_indices = cls_indices[:n_selected]
        n = len(cls_indices)
        n_train = int(round(n * train_ratio))
        n_val = int(round(n * val_ratio))
        n_test = n - n_train - n_val
        train_idx.extend(cls_indices[:n_train])
        val_idx.extend(cls_indices[n_train:n_train + n_val])
        test_idx.extend(cls_indices[n_train + n_val:n_train + n_val + n_test])
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)
    return np.array(train_idx), np.array(val_idx), np.array(test_idx)
def _print_split_stats(name: str, df: pd.DataFrame) -> None:
    total = len(df)
    label_counts = df["choice"].value_counts().sort_index().to_dict() if total > 0 else {}
    print(f"{name}: {total} rows | labels: {label_counts}")
def build_dataset_splits(triplets_csv: str, out_dir: str = "dataset/splits", seed: int = 42, train_ratio: float = 0.7, val_ratio: float = 0.15, test_ratio: float = 0.15) -> Dict[str, Path]:
    ratio_sum = train_ratio + val_ratio + test_ratio
    if abs(ratio_sum - 1.0) > 1e-6:
        raise ValueError(f"Ratios must sum to 1.0, got {ratio_sum}.")
    df = pd.read_csv(triplets_csv)
    _validate_triplet_columns(df)
    clean = pd.DataFrame({
        "reference": df["reference"].astype(str),
        "candidate_a": df["candidate_a"].astype(str),
        "candidate_b": df["candidate_b"].astype(str),
        "choice": _validate_binary_choice(df["choice"]),
    })
    train_idx, val_idx, test_idx = _stratified_indices(labels=clean["choice"].to_numpy(), train_ratio=train_ratio, val_ratio=val_ratio, test_ratio=test_ratio, seed=seed)
    train_df = clean.iloc[train_idx].reset_index(drop=True)
    val_df = clean.iloc[val_idx].reset_index(drop=True)
    test_df = clean.iloc[test_idx].reset_index(drop=True)
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    train_csv = out_path / "train.csv"
    val_csv = out_path / "val.csv"
    test_csv = out_path / "test.csv"
    train_df.to_csv(train_csv, index=False)
    val_df.to_csv(val_csv, index=False)
    test_df.to_csv(test_csv, index=False)
    _print_split_stats("train", train_df)
    _print_split_stats("val", val_df)
    _print_split_stats("test", test_df)
    return {"train": train_csv, "val": val_csv, "test": test_csv}
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
def prepare_triplets_csv(images_root: str) -> pd.DataFrame:
    root = Path(images_root)
    raw_csv = root / "data.csv"
    if not raw_csv.exists():
        raise FileNotFoundError(f"Missing raw metadata: {raw_csv}")
    df = pd.read_csv(raw_csv)
    for column in ["ref_path", "left_path", "right_path"]:
        df[column] = df[column].str.replace("util/2afc_src_images/", "", regex=False)
    df = df[df["left_vote"] != df["right_vote"]].copy()
    df["choice"] = (df["right_vote"] > df["left_vote"]).astype(int)
    triplets = pd.DataFrame({
        "reference": df["ref_path"],
        "candidate_a": df["left_path"],
        "candidate_b": df["right_path"],
        "choice": df["choice"],
    })
    exists = triplets.apply(lambda row: (root / row["reference"]).exists() and (root / row["candidate_a"]).exists() and (root / row["candidate_b"]).exists(), axis=1)

    # Filter
    print("Before filtering:", len(triplets))

    triplets = triplets[exists].reset_index(drop=True)

    print("After filtering:", len(triplets))
    print("Triplets shape:", triplets.shape)
    print("Triplets columns:", triplets.columns.tolist())
    print("First rows:")
    print(triplets.head())

    # Build output path
    out_csv = Path(CONFIG["triplets_csv"])
    print("Output path:", out_csv)

    # Save
    print("\n=== SAVING CSV ===")
    print("Attempting to save to:", out_csv)

    triplets.to_csv(out_csv, index=False)

    print("to_csv() completed")
    print("CSV exists:", out_csv.exists())

    if out_csv.exists():
        print("CSV size:", out_csv.stat().st_size, "bytes")
        print("Absolute path:", out_csv.resolve())
    else:
        print("!!! CSV WAS NOT CREATED !!!")
    return triplets
def batch_ranking_loss(sim_a: torch.Tensor, sim_b: torch.Tensor, label: torch.Tensor) -> torch.Tensor:
    sign = torch.where(label == 0, torch.ones_like(sim_a), -torch.ones_like(sim_a))
    margin = sim_a - sim_b
    return F.softplus(-sign * margin).mean()
def build_encoder_from_config(config: Dict) -> VisionTransformerEncoder:
    encoder = VisionTransformerEncoder(BackboneConfig(model_name=config["model_name"], pretrained=config["pretrained"], image_size=config["image_size"]))
    replace_layers = parse_replace_layers(config["replace_layers"])
    block_indices = resolve_block_indices(len(encoder.backbone.blocks), replace_layers)
    if config["attention_type"] != "standard":
        replace_attention_modules(encoder, make_attention_factory(config["attention_type"], top_k=config["top_k"]), block_indices=block_indices)
    return encoder
def build_optimizer_and_scheduler(encoder: nn.Module, config: Dict, steps_per_epoch: int, total_epochs: int):
    if config["attention_type"] in {"moh", "pyra", "meta"}:
        router_params = []
        base_params = []
        for name, param in encoder.named_parameters():
            if not param.requires_grad:
                continue
            if ".router." in name or ".meta_router." in name:
                router_params.append(param)
            else:
                base_params.append(param)
        param_groups = [{"params": base_params, "lr": config["lr"], "weight_decay": config["weight_decay"]}]
        if router_params:
            param_groups.append({"params": router_params, "lr": config["router_lr"], "weight_decay": config["weight_decay"]})
        optimizer = torch.optim.AdamW(param_groups)
    else:
        optimizer = torch.optim.AdamW(encoder.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    total_steps = max(1, total_epochs * max(1, steps_per_epoch))
    warmup_steps = int(total_steps * config["warmup_ratio"])
    def lr_lambda(step: int) -> float:
        if warmup_steps > 0 and step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        cosine = 0.5 * (1.0 + np.cos(np.pi * progress))
        floor = config["min_lr"] / max(config["lr"], 1e-12)
        return max(floor, cosine)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    return optimizer, scheduler
def evaluate_loader(pipeline: DreamSimLikePipeline, loader: DataLoader, device: str, amp_enabled: bool, criterion):
    pipeline.eval()
    total_loss = 0.0
    total = 0
    correct = 0
    with torch.no_grad():
        for batch in loader:
            ref = batch["reference"].to(device, non_blocking=True)
            a = batch["candidate_a"].to(device, non_blocking=True)
            b = batch["candidate_b"].to(device, non_blocking=True)
            label = batch["choice"].to(device, non_blocking=True)
            with autocast(enabled=amp_enabled):
                out = pipeline(ref, a, b)
                loss = criterion(out["sim_a"], out["sim_b"], label)
            total_loss += loss.item() * label.numel()
            total += label.numel()
            correct += (out["pred"] == label).sum().item()
    return total_loss / max(total, 1), correct / max(total, 1)
EXPERIMENTS: List[Dict] = [
    {"name": "baseline_deit_small", "model_name": "deit_small_patch16_224", "pretrained": False, "attention_type": "standard", "replace_layers": None},
    {"name": "moh_all_deit_small", "model_name": "deit_small_patch16_224", "pretrained": False, "attention_type": "moh", "replace_layers": "all"},
    {"name": "moh_last6_deit_small", "model_name": "deit_small_patch16_224", "pretrained": False, "attention_type": "moh", "replace_layers": [6, 7, 8, 9, 10, 11]},
    {"name": "moh_first6_deit_small", "model_name": "deit_small_patch16_224", "pretrained": False, "attention_type": "moh", "replace_layers": [0, 1, 2, 3, 4, 5]},
    {"name": "moh_every_other_deit_small", "model_name": "deit_small_patch16_224", "pretrained": False, "attention_type": "moh", "replace_layers": [0, 2, 4, 6, 8, 10]},
]
def format_replace_layers(value: Optional[object]) -> str:
    if value is None:
        return "none"
    if value == "all":
        return "all"
    if isinstance(value, list):
        return ",".join(str(item) for item in value)
    return str(value)

In [6]:
class MoHAttention(nn.Module):
    def __init__(self, old_attn: nn.Module, top_k: int | None = 4):
        super().__init__()
        self.num_heads = old_attn.num_heads
        self.head_dim = old_attn.head_dim
        self.attn_dim = old_attn.attn_dim
        self.scale = old_attn.scale
        self.qkv = copy.deepcopy(old_attn.qkv)
        self.q_norm = copy.deepcopy(old_attn.q_norm)
        self.k_norm = copy.deepcopy(old_attn.k_norm)
        self.attn_drop = copy.deepcopy(old_attn.attn_drop)
        self.norm = copy.deepcopy(old_attn.norm)
        self.proj = copy.deepcopy(old_attn.proj)
        self.proj_drop = copy.deepcopy(old_attn.proj_drop)
        dim = old_attn.qkv.in_features
        self.router = nn.Linear(dim, self.num_heads)
        self.top_k = top_k
    def forward(self, x, attn_mask=None, is_causal=False):
        batch_size, seq_len, _ = x.shape
        qkv = self.qkv(x).reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        q = self.q_norm(q)
        k = self.k_norm(k)
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)
        if attn_mask is not None:
            attn = attn + attn_mask
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        head_out = attn @ v
        route_logits = self.router(x.mean(dim=1))
        if self.top_k is not None and self.top_k < self.num_heads:
            topk_vals, topk_idx = route_logits.topk(self.top_k, dim=-1)
            masked_logits = route_logits.new_full(route_logits.shape, float("-inf"))
            masked_logits.scatter_(1, topk_idx, topk_vals)
            route_logits = masked_logits
        head_weights = route_logits.softmax(dim=-1)
        head_out = head_out * head_weights[:, :, None, None]
        out = head_out.transpose(1, 2).reshape(batch_size, seq_len, self.attn_dim)
        out = self.norm(out)
        out = self.proj(out)
        out = self.proj_drop(out)
        return out
class PyraAttention(nn.Module):
    def __init__(self, old_attn: nn.Module, top_k: int | None = 4):
        super().__init__()
        self.num_heads = old_attn.num_heads
        self.head_dim = old_attn.head_dim
        self.attn_dim = old_attn.attn_dim
        self.scale = old_attn.scale
        self.qkv = copy.deepcopy(old_attn.qkv)
        self.q_norm = copy.deepcopy(old_attn.q_norm)
        self.k_norm = copy.deepcopy(old_attn.k_norm)
        self.attn_drop = copy.deepcopy(old_attn.attn_drop)
        self.norm = copy.deepcopy(old_attn.norm)
        self.proj = copy.deepcopy(old_attn.proj)
        self.proj_drop = copy.deepcopy(old_attn.proj_drop)
        dim = old_attn.qkv.in_features
        self.router = nn.Linear(dim, self.num_heads)
        self.top_k = top_k

    def forward(self, x, attn_mask=None, is_causal=False):
        bsz, seq_len, _ = x.shape
        qkv = self.qkv(x).reshape(bsz, seq_len, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        q = self.q_norm(q)
        k = self.k_norm(k)
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)
        if attn_mask is not None:
            attn = attn + attn_mask
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        head_out = attn @ v
        route_logits = self.router(x.mean(dim=1))
        head_prior = torch.linspace(1.0, 0.0, self.num_heads, device=route_logits.device, dtype=route_logits.dtype)
        route_logits = route_logits + head_prior.unsqueeze(0)
        if self.top_k is not None and self.top_k < self.num_heads:
            topk_vals, topk_idx = route_logits.topk(self.top_k, dim=-1)
            masked_logits = route_logits.new_full(route_logits.shape, float("-inf"))
            masked_logits.scatter_(1, topk_idx, topk_vals)
            route_logits = masked_logits
        head_weights = route_logits.softmax(dim=-1)
        head_out = head_out * head_weights[:, :, None, None]
        out = head_out.transpose(1, 2).reshape(bsz, seq_len, self.attn_dim)
        out = self.norm(out)
        out = self.proj(out)
        out = self.proj_drop(out)
        return out


class MetaAttention(nn.Module):
    def __init__(self, old_attn: nn.Module, top_k: int | None = 4):
        super().__init__()
        self.num_heads = old_attn.num_heads
        self.head_dim = old_attn.head_dim
        self.attn_dim = old_attn.attn_dim
        self.scale = old_attn.scale
        self.qkv = copy.deepcopy(old_attn.qkv)
        self.q_norm = copy.deepcopy(old_attn.q_norm)
        self.k_norm = copy.deepcopy(old_attn.k_norm)
        self.attn_drop = copy.deepcopy(old_attn.attn_drop)
        self.norm = copy.deepcopy(old_attn.norm)
        self.proj = copy.deepcopy(old_attn.proj)
        self.proj_drop = copy.deepcopy(old_attn.proj_drop)
        dim = old_attn.qkv.in_features
        hidden = max(32, dim // 4)
        self.meta_router = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, self.num_heads))
        self.top_k = top_k

    def forward(self, x, attn_mask=None, is_causal=False):
        bsz, seq_len, _ = x.shape
        qkv = self.qkv(x).reshape(bsz, seq_len, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        q = self.q_norm(q)
        k = self.k_norm(k)
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)
        if attn_mask is not None:
            attn = attn + attn_mask
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        head_out = attn @ v
        route_logits = self.meta_router(x.mean(dim=1))
        if self.top_k is not None and self.top_k < self.num_heads:
            topk_vals, topk_idx = route_logits.topk(self.top_k, dim=-1)
            masked_logits = route_logits.new_full(route_logits.shape, float("-inf"))
            masked_logits.scatter_(1, topk_idx, topk_vals)
            route_logits = masked_logits
        head_weights = route_logits.softmax(dim=-1)
        head_out = head_out * head_weights[:, :, None, None]
        out = head_out.transpose(1, 2).reshape(bsz, seq_len, self.attn_dim)
        out = self.norm(out)
        out = self.proj(out)
        out = self.proj_drop(out)
        return out


def make_attention_factory(attention_type: str, top_k: int | None = 4):
    if attention_type == "standard":
        return lambda old_attn, block_idx: old_attn
    if attention_type == "moh":
        return lambda old_attn, block_idx: MoHAttention(old_attn, top_k=top_k)
    if attention_type == "pyra":
        return lambda old_attn, block_idx: PyraAttention(old_attn, top_k=top_k)
    if attention_type == "meta":
        return lambda old_attn, block_idx: MetaAttention(old_attn, top_k=top_k)
    raise ValueError(f"Unknown attention_type: {attention_type}")


def build_optimizer_and_scheduler(encoder: nn.Module, config: Dict, steps_per_epoch: int, total_epochs: int):
    if config["attention_type"] in {"moh", "pyra", "meta"}:
        router_params = []
        base_params = []
        for name, param in encoder.named_parameters():
            if not param.requires_grad:
                continue
            if ".router." in name or ".meta_router." in name:
                router_params.append(param)
            else:
                base_params.append(param)
        param_groups = [{"params": base_params, "lr": config["lr"], "weight_decay": config["weight_decay"]}]
        if router_params:
            param_groups.append({"params": router_params, "lr": config["router_lr"], "weight_decay": config["weight_decay"]})
        optimizer = torch.optim.AdamW(param_groups)
    else:
        optimizer = torch.optim.AdamW(encoder.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    total_steps = max(1, total_epochs * max(1, steps_per_epoch))
    warmup_steps = int(total_steps * config["warmup_ratio"])

    def lr_lambda(step: int) -> float:
        if warmup_steps > 0 and step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        cosine = 0.5 * (1.0 + np.cos(np.pi * progress))
        floor = config["min_lr"] / max(config["lr"], 1e-12)
        return max(floor, cosine)

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    return optimizer, scheduler

# Data

In [ ]:
# results will be saved in /content/drive/MyDrive/CV/dataset/processed/triplets_available.csv
seed_everything(CONFIG["seed"])
triplets = prepare_triplets_csv(CONFIG["images_root"])

Before filtering: 64420
After filtering: 20019
Triplets shape: (20019, 4)
Triplets columns: ['reference', 'candidate_a', 'candidate_b', 'choice']
First rows:
         reference            candidate_a            candidate_b  choice
0  ref/000/002.png  distort/000/002_0.png  distort/000/002_1.png       0
1  ref/000/006.png  distort/000/006_1.png  distort/000/006_0.png       0
2  ref/000/017.png  distort/000/017_1.png  distort/000/017_0.png       0
3  ref/000/026.png  distort/000/026_0.png  distort/000/026_1.png       1
4  ref/000/032.png  distort/000/032_0.png  distort/000/032_1.png       0
Output path: /content/drive/MyDrive/CV/dataset/processed/triplet_available.csv

=== SAVING CSV ===
Attempting to save to: /content/drive/MyDrive/CV/dataset/processed/triplet_available.csv
to_csv() completed
CSV exists: True
CSV size: 1241219 bytes
Absolute path: /content/drive/MyDrive/CV/dataset/processed/triplet_available.csv


In [ ]:
print("usable triplets:", len(triplets))
print(triplets["choice"].value_counts(dropna=False))

usable triplets: 20019
choice
1    10167
0     9852
Name: count, dtype: int64


In [ ]:
# results will be saved in /content/drive/MyDrive/CV/dataset/splits/train.csv .../val.csv .../test.csv
splits_dir = Path(BASE_DIR, "dataset/splits")
if not (splits_dir / "train.csv").exists() or not (splits_dir / "val.csv").exists() or not (splits_dir / "test.csv").exists():
    print("Building splits from dataset/processed/triplets_available.csv ...")
    build_dataset_splits(triplets_csv=CONFIG["triplets_csv"], out_dir=str(splits_dir), seed=CONFIG["seed"], train_ratio=0.7, val_ratio=0.15, test_ratio=0.15)

Building splits from dataset/processed/triplets_available.csv ...
train: 2802 rows | labels: {0: 1379, 1: 1423}
val: 601 rows | labels: {0: 296, 1: 305}
test: 600 rows | labels: {0: 295, 1: 305}


In [9]:
train_ds = NightsTripletDataset(split_csv=CONFIG["train_csv"], images_root=CONFIG["images_root"], transform=build_transforms(image_size=CONFIG["image_size"], train=True))
val_ds = NightsTripletDataset(split_csv=CONFIG["val_csv"], images_root=CONFIG["images_root"], transform=build_transforms(image_size=CONFIG["image_size"], train=False))
test_ds = NightsTripletDataset(split_csv=CONFIG["test_csv"], images_root=CONFIG["images_root"], transform=build_transforms(image_size=CONFIG["image_size"], train=False))

In [10]:
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=CONFIG["device"].startswith("cuda"))
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=CONFIG["device"].startswith("cuda"))
print("train/val/test sizes:", len(train_ds), len(val_ds), len(test_ds))

train/val/test sizes: 2802 601 600


# Network

Training Process:
1. Manually modify the Config in Globals section: "attention_type", "top_k", "replace_layers".  
2. Execute Network and Train section.
3. Manually modify the saving path in Save Results and run.

In [11]:
encoder = build_encoder_from_config(CONFIG).to(CONFIG["device"])
pipeline = DreamSimLikePipeline(encoder).to(CONFIG["device"])
optimizer, scheduler = build_optimizer_and_scheduler(encoder=encoder, config=CONFIG, steps_per_epoch=len(train_loader), total_epochs=CONFIG["epochs"])
criterion = batch_ranking_loss
amp_enabled = CONFIG["device"].startswith("cuda")
scaler = GradScaler(enabled=amp_enabled)
print("Model ready on", CONFIG["device"])
print("attention_type:", CONFIG["attention_type"], "replace_layers:", CONFIG["replace_layers"], "top_k:", CONFIG["top_k"])

Model ready on cuda
attention_type: moh replace_layers: all top_k: 4


/tmp/ipykernel_1500/1774775247.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=amp_enabled)


# Train

In [ ]:
history = []
best_acc = -1.0
best_state = None

print("=" * 80)
print("START TRAINING")
print(f"Device: {CONFIG['device']}")
print(f"Epochs: {CONFIG['epochs']}")
print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"AMP enabled: {amp_enabled}")
print("=" * 80)

for epoch in range(1, CONFIG["epochs"] + 1):
    start = time.perf_counter()

    print(f"\n{'=' * 30} EPOCH {epoch}/{CONFIG['epochs']} {'=' * 30}")
    print("Setting model to train mode...")
    pipeline.train()

    train_loss_sum = 0.0
    train_count = 0

    epoch_batch_start = time.perf_counter()

    for batch_idx, batch in enumerate(train_loader, start=1):

        # Print progress every 10 batches
        if batch_idx == 1 or batch_idx % 10 == 0 or batch_idx == len(train_loader):
            elapsed = time.perf_counter() - epoch_batch_start
            print(
                f"[Epoch {epoch}] "
                f"batch {batch_idx}/{len(train_loader)} "
                f"({100 * batch_idx / len(train_loader):.1f}%) "
                f"elapsed={elapsed:.1f}s"
            )

        # ---------------------------------------------------------
        # Move data to GPU
        # ---------------------------------------------------------
        ref = batch["reference"].to(
            CONFIG["device"], non_blocking=True
        )
        a = batch["candidate_a"].to(
            CONFIG["device"], non_blocking=True
        )
        b = batch["candidate_b"].to(
            CONFIG["device"], non_blocking=True
        )
        label = batch["choice"].to(
            CONFIG["device"], non_blocking=True
        )

        # Debug first batch
        if batch_idx == 1:
            print(
                f"  Batch size: {label.numel()}"
            )
            print(
                f"  ref shape: {ref.shape}"
            )
            print(
                f"  candidate_a shape: {a.shape}"
            )
            print(
                f"  candidate_b shape: {b.shape}"
            )
            print(
                f"  label shape: {label.shape}"
            )

        # ---------------------------------------------------------
        # Forward + loss
        # ---------------------------------------------------------
        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=amp_enabled):
            out = pipeline(ref, a, b)
            loss = criterion(
                out["sim_a"],
                out["sim_b"],
                label
            )

        # Debug first batch
        if batch_idx == 1:
            print(f"  First batch loss: {loss.item():.6f}")
            print(
                f"  sim_a shape: {out['sim_a'].shape}"
            )
            print(
                f"  sim_b shape: {out['sim_b'].shape}"
            )

        # ---------------------------------------------------------
        # Backward + optimizer
        # ---------------------------------------------------------
        scaler.scale(loss).backward()

        scaler.step(optimizer)
        scaler.update()

        scheduler.step()

        # ---------------------------------------------------------
        # Accumulate loss
        # ---------------------------------------------------------
        train_loss_sum += loss.item() * label.numel()
        train_count += label.numel()

    train_loss = train_loss_sum / max(train_count, 1)

    train_seconds = time.perf_counter() - epoch_batch_start

    print(
        f"\nEpoch {epoch} training finished."
    )
    print(
        f"  Train samples: {train_count}"
    )
    print(
        f"  Train loss: {train_loss:.6f}"
    )
    print(
        f"  Training time: {train_seconds:.2f}s"
    )

    # -------------------------------------------------------------
    # Validation
    # -------------------------------------------------------------
    print(f"\nStarting validation for epoch {epoch}...")

    val_start = time.perf_counter()

    val_loss, val_acc = evaluate_loader(
        pipeline=pipeline,
        loader=val_loader,
        device=CONFIG["device"],
        amp_enabled=amp_enabled,
        criterion=criterion
    )

    val_seconds = time.perf_counter() - val_start

    print(
        f"Validation finished in {val_seconds:.2f}s"
    )
    print(
        f"  val_loss = {val_loss:.6f}"
    )
    print(
        f"  val_accuracy_2afc = {val_acc:.4f}"
    )

    # -------------------------------------------------------------
    # Epoch information
    # -------------------------------------------------------------
    epoch_seconds = time.perf_counter() - start

    current_lr = optimizer.param_groups[0]["lr"]

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy_2afc": val_acc,
        "epoch_seconds": epoch_seconds,
        "lr": current_lr,
    }

    history.append(row)

    print("\n" + "-" * 70)
    print(
        f"EPOCH {epoch} SUMMARY"
    )
    print(
        f"  train_loss = {train_loss:.6f}"
    )
    print(
        f"  val_loss   = {val_loss:.6f}"
    )
    print(
        f"  val_acc    = {val_acc:.4f}"
    )
    print(
        f"  lr         = {current_lr:.2e}"
    )
    print(
        f"  time       = {epoch_seconds:.2f}s"
    )
    print("-" * 70)

    # -------------------------------------------------------------
    # Best model
    # -------------------------------------------------------------
    if val_acc > best_acc:

        print(
            f"\n*** NEW BEST MODEL ***"
        )
        print(
            f"Previous best accuracy: {best_acc:.4f}"
        )
        print(
            f"New best accuracy:      {val_acc:.4f}"
        )

        best_acc = val_acc

        best_state = {
            "epoch": epoch,
            "model_state_dict": pipeline.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "config": CONFIG,
            "best_val_accuracy_2afc": best_acc,
        }

    else:
        print(
            f"\nNo improvement. "
            f"Best accuracy remains {best_acc:.4f}"
        )

START TRAINING
Device: cuda
Epochs: 3
Train batches: 176
Validation batches: 38
AMP enabled: True

============================== EPOCH 1/3 ==============================
Setting model to train mode...
[Epoch 1] batch 1/176 (0.6%) elapsed=46.3s
  Batch size: 16
  ref shape: torch.Size([16, 3, 224, 224])
  candidate_a shape: torch.Size([16, 3, 224, 224])
  candidate_b shape: torch.Size([16, 3, 224, 224])
  label shape: torch.Size([16])


/tmp/ipykernel_1500/1967165338.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=amp_enabled):


  First batch loss: 0.663543
  sim_a shape: torch.Size([16])
  sim_b shape: torch.Size([16])
[Epoch 1] batch 10/176 (5.7%) elapsed=147.8s
[Epoch 1] batch 20/176 (11.4%) elapsed=273.6s
[Epoch 1] batch 30/176 (17.0%) elapsed=394.0s
[Epoch 1] batch 40/176 (22.7%) elapsed=517.6s
[Epoch 1] batch 50/176 (28.4%) elapsed=643.5s
[Epoch 1] batch 60/176 (34.1%) elapsed=766.9s
[Epoch 1] batch 70/176 (39.8%) elapsed=889.2s
[Epoch 1] batch 80/176 (45.5%) elapsed=1011.1s
[Epoch 1] batch 90/176 (51.1%) elapsed=1133.9s
[Epoch 1] batch 100/176 (56.8%) elapsed=1254.8s
[Epoch 1] batch 110/176 (62.5%) elapsed=1377.4s
[Epoch 1] batch 120/176 (68.2%) elapsed=1498.8s
[Epoch 1] batch 130/176 (73.9%) elapsed=1621.0s


In [ ]:
# =================================================================
# SAVE RESULTS
# =================================================================

print("\n" + "=" * 80)
print("TRAINING FINISHED")
print("=" * 80)

print(f"Best validation accuracy: {best_acc:.4f}")
print(f"Total epochs completed: {CONFIG['epochs']}")

run_name = CONFIG["run_name"]

# manually modify these three lines according to the training executed above.
# best_template_run.pt              train_history_template_run.json
# moh_all_best_template_run.pt      moh_all_train_history_template_run.json
# moh_first_best_template_run.pt    moh_first_train_history_template_run.json
# moh_last_best_template_run.pt     moh_last_train_history_template_run.json
# moh_every_best_template_run.pt    moh_every_train_history_template_run.json
# pyra_every_best_template_run.pt   pyra_every_train_history_template_run.json
# meta_every_best_template_run.pt   meta_every_train_history_template_run.json
history_path = Path(
    CONFIG["out_dir"], "logs", f"final_meta_every_train_history_{run_name}.json"
)

best_path = Path(
    CONFIG["out_dir"], "checkpoints", f"final_meta_every_best_{run_name}.pt"
)

if best_state is not None:

    print(f"\nSaving best model...")
    print(f"  Best model path: {best_path}")

    torch.save(
        best_state,
        best_path
    )

    print("  Best model saved successfully.")

else:
    print("\nWARNING: best_state is None!")
    print("Best model was NOT saved.")


print("\nSaving training history...")
print(f"  History path: {history_path}")

history_path.write_text(
    json.dumps(history, indent=2),
    encoding="utf-8"
)

print("  Training history saved successfully.")
print("=" * 80)


TRAINING FINISHED
Best validation accuracy: 0.8669
Total epochs completed: 3

Saving final model...
  Last model path:  /content/drive/MyDrive/CV/outputs/checkpoints/final_meta_every_last_template_run.pt
  Last model saved successfully.

Saving best model...
  Best model path: /content/drive/MyDrive/CV/outputs/checkpoints/final_meta_every_best_template_run.pt
  Best model saved successfully.

Saving training history...
  History path: /content/drive/MyDrive/CV/outputs/logs/final_meta_every_train_history_template_run.json
  Training history saved successfully.

ALL FILES SAVED
History : /content/drive/MyDrive/CV/outputs/logs/final_meta_every_train_history_template_run.json
Best    : /content/drive/MyDrive/CV/outputs/checkpoints/final_meta_every_best_template_run.pt
Last    : /content/drive/MyDrive/CV/outputs/checkpoints/final_meta_every_last_template_run.pt


# Test

In [ ]:
def build_variant_encoder(model_name: str, image_size: int, attention_type: str, replace_layers, top_k):
    variant_encoder = VisionTransformerEncoder(BackboneConfig(model_name=model_name, pretrained=False, image_size=image_size))
    parsed_layers = parse_replace_layers(replace_layers) if isinstance(replace_layers, str) else replace_layers
    block_indices = resolve_block_indices(len(variant_encoder.backbone.blocks), parsed_layers)
    if attention_type != "standard":
        replace_attention_modules(variant_encoder, make_attention_factory(attention_type, top_k=top_k), block_indices=block_indices)
    return variant_encoder

def evaluate_checkpoint_variant(cfg: Dict, split_csv: str, images_root: str, image_size: int, batch_size: int, num_workers: int, device: str) -> List[Dict]:
    rows = []
    untrained_encoder = build_variant_encoder(CONFIG["model_name"], image_size, cfg["attention_type"], cfg["replace_layers"], cfg["top_k"])
    untrained_metrics = evaluate(split_csv=split_csv, images_root=images_root, model_name=CONFIG["model_name"], pretrained=False, image_size=image_size, batch_size=batch_size, num_workers=num_workers, device=device, encoder=untrained_encoder)
    rows.append({"name": cfg["name"], "state": "untrained", **untrained_metrics})

    trained_encoder = build_variant_encoder(CONFIG["model_name"], image_size, cfg["attention_type"], cfg["replace_layers"], cfg["top_k"])
    trained_pipeline = DreamSimLikePipeline(trained_encoder)
    checkpoint = torch.load(cfg["path"],map_location=CONFIG["device"],weights_only=False)
    state_dict = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
    trained_pipeline.load_state_dict(state_dict)
    trained_metrics = evaluate(split_csv=split_csv, images_root=images_root, model_name=CONFIG["model_name"], pretrained=False, image_size=image_size, batch_size=batch_size, num_workers=num_workers, device=device, encoder=trained_pipeline.encoder)
    rows.append({"name": cfg["name"], "state": "trained", **trained_metrics})
    return rows

# comparison baseline/moh/pyra/meta
CHECKPOINT_MODELS = [
    {"name": "baseline", "path": f"{CONFIG['out_dir']}/checkpoints/best_template_run.pt", "attention_type": "standard", "replace_layers": "none", "top_k": None},
    {"name": "moh_every", "path": f"{CONFIG['out_dir']}/checkpoints/final_moh_every_best_template_run.pt", "attention_type": "moh", "replace_layers": "0, 2, 4, 6, 8, 10", "top_k": 4},
    {"name": "pyra_every", "path": f"{CONFIG['out_dir']}/checkpoints/final_pyra_every_best_template_run.pt", "attention_type": "pyra", "replace_layers": "0, 2, 4, 6, 8, 10", "top_k": 4},
    {"name": "meta_every", "path": f"{CONFIG['out_dir']}/checkpoints/final_meta_every_best_template_run.pt", "attention_type": "meta", "replace_layers": "0, 2, 4, 6, 8, 10", "top_k": 4},
]

checkpoint_rows: List[Dict] = []
for cfg in CHECKPOINT_MODELS:
    checkpoint_rows.extend(evaluate_checkpoint_variant(cfg, split_csv=CONFIG["test_csv"], images_root=CONFIG["images_root"], image_size=CONFIG["image_size"], batch_size=CONFIG["batch_size"], num_workers=CONFIG["num_workers"], device=CONFIG["device"]))

checkpoint_results_df = pd.DataFrame(checkpoint_rows)
checkpoint_results_csv = Path(CONFIG["out_dir"]) / "tables" / "checkpoint_test_results.csv"
checkpoint_results_csv.parent.mkdir(parents=True, exist_ok=True)
checkpoint_results_df.to_csv(checkpoint_results_csv, index=False)
print(checkpoint_results_df)

# comparison baseline/moh_all/moh_first/moh_last/moh_every
CHECKPOINT_MODELS = [
    {"name": "baseline", "path": f"{CONFIG['out_dir']}/checkpoints/best_template_run.pt", "attention_type": "standard", "replace_layers": "none", "top_k": None},
    {"name": "moh_all", "path": f"{CONFIG['out_dir']}/checkpoints/moh_all_best_template_run.pt", "attention_type": "moh", "replace_layers": "all", "top_k": 4},
    {"name": "moh_first", "path": f"{CONFIG['out_dir']}/checkpoints/moh_first_best_template_run.pt", "attention_type": "moh", "replace_layers": "0, 1, 2, 3, 4, 5", "top_k": 4},
    {"name": "moh_last", "path": f"{CONFIG['out_dir']}/checkpoints/moh_last_best_template_run.pt", "attention_type": "moh", "replace_layers": "6, 7, 8, 9, 10, 11", "top_k": 4},
    {"name": "moh_every", "path": f"{CONFIG['out_dir']}/checkpoints/moh_every_best_template_run.pt", "attention_type": "moh", "replace_layers": "0, 2, 4, 6, 8, 10", "top_k": 4},
]

checkpoint_rows: List[Dict] = []
for cfg in CHECKPOINT_MODELS:
    checkpoint_rows.extend(evaluate_checkpoint_variant(cfg, split_csv=CONFIG["test_csv"], images_root=CONFIG["images_root"], image_size=CONFIG["image_size"], batch_size=CONFIG["batch_size"], num_workers=CONFIG["num_workers"], device=CONFIG["device"]))

checkpoint_results_df = pd.DataFrame(checkpoint_rows)
checkpoint_results_csv = Path(CONFIG["out_dir"]) / "tables" / "checkpoint_test_results.csv"
checkpoint_results_csv.parent.mkdir(parents=True, exist_ok=True)
checkpoint_results_df.to_csv(checkpoint_results_csv, mode="a", header=False,index=False)
print(checkpoint_results_df)


         name      state  samples  batches  accuracy_2afc  ms_per_image  \
0    baseline  untrained      600       38       0.636667      4.456327   
1    baseline    trained      600       38       0.848333      3.612733   
2   moh_every  untrained      600       38       0.640000      4.005067   
3   moh_every    trained      600       38       0.871667      3.757408   
4  pyra_every  untrained      600       38       0.640000      3.828232   
5  pyra_every    trained      600       38       0.871667      3.746938   
6  meta_every  untrained      600       38       0.651667      3.834442   
7  meta_every    trained      600       38       0.876667      3.676470   

   images_per_second  elapsed_seconds  max_vram_mb  
0         224.400060         8.021388   174.103027  
1         276.798746         6.502920   508.338379  
2         249.683734         7.209120   508.338379  
3         266.140900         6.763335   513.891113  
4         261.217186         6.890818   513.891113  
5     

In [7]:
# code integrated in the above cell

        name      state  samples  batches  accuracy_2afc  ms_per_image  \
0   baseline  untrained      600       38       0.648333      1.139843   
1   baseline    trained      600       38       0.848333      0.908724   
2    moh_all  untrained      600       38       0.618333      1.058058   
3    moh_all    trained      600       38       0.755000      1.075068   
4  moh_first  untrained      600       38       0.640000      0.963745   
5  moh_first    trained      600       38       0.805000      0.971499   
6   moh_last  untrained      600       38       0.643333      0.967939   
7   moh_last    trained      600       38       0.811667      1.031071   
8  moh_every  untrained      600       38       0.635000      0.979926   
9  moh_every    trained      600       38       0.840000      0.982369   

   images_per_second  elapsed_seconds  max_vram_mb  
0         877.313799         2.051717   174.103027  
1        1100.444065         1.635703   508.338379  
2         945.127535      

draw figures in vscode with python script and write report in google docs!